In [1]:
!nvidia-smi

In [2]:
import numpy as np
import torch
import torch.nn as nn
import tiktoken
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

In [3]:
tokenizer = tiktoken.get_encoding("gpt2")

In [4]:
gpt_config_small = {
    "vocab_size": 50257,
    "emb_dim": 256,
    "context_length": 1028,
    "num_heads": 4,
    "n_layers": 8,
    "n_attn": 4,
    "drop_rate": 0.2,
    "qkv_bias": False
}

cfg = gpt_config_small

In [5]:
TOKEN_FILE = Path(r"D:\built a llm from scratch 124million parameters\fineweb-edu\fineweb_tokens.bin")
token_ids = np.memmap(TOKEN_FILE, dtype=np.uint16, mode="r")

VAL_START, VAL_END = 3_000_000_000, 3_003_000_000
SIZES = {"5M": 5_000_000, "25M": 25_000_000, "100M": 100_000_000}

TOKEN_BUDGET = 100_000_000
SEED = 123
MICRO_BS = 4
ACCUM = 4

print("Total tokens in file:", f"{len(token_ids):,}")

Total tokens in file: 3,003,000,000


In [6]:
class FineWebTokenDataset(Dataset):
    def __init__(self, token_ids, start_idx, end_idx, seq_len):
        self.token_ids = token_ids
        self.start_idx = start_idx
        self.end_idx = min(end_idx, len(token_ids))
        self.seq_len = seq_len
        self.n_samples = max(0, (self.end_idx - self.start_idx - 1) // self.seq_len)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = self.start_idx + idx * self.seq_len
        x = np.asarray(self.token_ids[start:start + self.seq_len], dtype=np.int64).copy()
        y = np.asarray(self.token_ids[start + 1:start + self.seq_len + 1], dtype=np.int64).copy()
        return torch.from_numpy(x), torch.from_numpy(y)

In [7]:
def make_loaders(size_name, seq_len=cfg["context_length"], micro_bs=MICRO_BS, seed=SEED):
    train_ds = FineWebTokenDataset(token_ids, 0, SIZES[size_name], seq_len)
    val_ds = FineWebTokenDataset(token_ids, VAL_START, VAL_END, seq_len)
    gen = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=micro_bs, shuffle=True,
                              drop_last=True, generator=gen, num_workers=0)
    train_eval_loader = DataLoader(train_ds, batch_size=micro_bs, shuffle=False,
                                   drop_last=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=micro_bs, shuffle=False,
                            drop_last=False, num_workers=0)
    return train_loader, train_eval_loader, val_loader

In [8]:
for name, n in SIZES.items():
    tl, tel, vl = make_loaders(name)
    total_updates = TOKEN_BUDGET // (MICRO_BS * ACCUM * cfg["context_length"])
    print(f"{name:>4} | train seqs {len(tl.dataset):>7,} | val seqs {len(vl.dataset):,} | "
          f"epochs {TOKEN_BUDGET / n:5.1f} | updates {total_updates:,}")

x, y = next(iter(tl))
print("batch shape:", x.shape, y.shape)

  5M | train seqs   4,863 | val seqs 2,918 | epochs  20.0 | updates 6,079
 25M | train seqs  24,319 | val seqs 2,918 | epochs   4.0 | updates 6,079
100M | train seqs  97,276 | val seqs 2,918 | epochs   1.0 | updates 6,079
batch shape: torch.Size([4, 1028]) torch.Size([4, 1028])


In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.w_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.w_values = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.outproj = nn.Linear(d_out, d_out)

    def forward(self, x):
        b, num_tokens, din = x.shape

        keys = self.w_key(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = self.w_values(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        query = self.w_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        context_vec = torch.nn.functional.scaled_dot_product_attention(
            query, keys, values,
            dropout_p=self.dropout.p if self.training else 0.0,
            is_causal=True
        ).transpose(1, 2)

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        return self.outproj(context_vec)

In [10]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean_x = x.mean(dim=-1, keepdim=True)
        var_x = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean_x) / torch.sqrt(var_x + self.eps)
        return self.scale * norm_x + self.shift

In [11]:
class Gelu(nn.Module):
    def forward(self, x):
        return torch.nn.functional.gelu(x)

In [12]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            Gelu(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layer(x)

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg, n_attn=4):
        super().__init__()
        self.attns = nn.ModuleList([
            MultiHeadAttention(
                d_in=cfg['emb_dim'],
                d_out=cfg['emb_dim'],
                dropout=cfg['drop_rate'],
                num_heads=cfg['num_heads'],
                qkv_bias=cfg['qkv_bias']
            )
            for _ in range(n_attn)
        ])
        self.attn_norm = LayerNorm(cfg['emb_dim'])
        self.ff = FeedForward(cfg)
        self.ff_norm = LayerNorm(cfg['emb_dim'])
        self.dropout = nn.Dropout(cfg['drop_rate'])

    def forward(self, x):
        shortcut = x
        h = self.attn_norm(x)
        for attn in self.attns:      # MHA -> MHA -> MHA -> MHA
            h = attn(h)
        x = shortcut + self.dropout(h)

        shortcut = x
        x = self.ff(self.ff_norm(x))
        x = shortcut + self.dropout(x)
        return x

In [14]:
class Gptmodel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.token_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_embed = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.dropout_layer = nn.Dropout(cfg["drop_rate"])

        self.transformer = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_LN = LayerNorm(cfg["emb_dim"])
        self.out_proj = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_id):
        batch_size, seq_len = in_id.shape

        token_em = self.token_emb(in_id)
        pos_em = self.pos_embed(torch.arange(seq_len, device=in_id.device))

        input_embed = token_em + pos_em
        input_embed = self.dropout_layer(input_embed)
        input_embed = self.transformer(input_embed)
        input_embed = self.final_LN(input_embed)
        return self.out_proj(input_embed)

In [15]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)

    return idx


def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    return torch.tensor(encoded).unsqueeze(0)


def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

In [16]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.float().flatten(0, 1), target_batch.flatten()
    )
    return loss


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.

    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break

    return total_loss / num_batches

In [17]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(
            train_loader, model, device, num_batches=eval_iter
        )
        val_loss = calc_loss_loader(
            val_loader, model, device, num_batches=eval_iter
        )
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = model.pos_embed.weight.shape[0]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)
    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model, idx=encoded,
            max_new_tokens=50, context_size=context_size
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()

In [18]:
from tqdm.auto import tqdm
import time

def train_model_simple(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    num_epochs,
    eval_freq,
    eval_iter,
    start_context,
    tokenizer,
    checkpoint_freq=10_000,
    run_name="run"
):
    train_losses = []
    val_losses = []
    track_tokens_seen = []

    tokens_seen = 0
    global_step = -1

    total_steps = num_epochs * len(train_loader)
    start_time = time.time()

    progress_bar = tqdm(
        total=total_steps,
        desc="Pretraining",
        unit="step"
    )

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()

            loss = calc_loss_batch(
                input_batch,
                target_batch,
                model,
                device
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            progress_bar.update(1)

            elapsed = time.time() - start_time
            completed = global_step + 1
            remaining = (
                elapsed / completed * (total_steps - completed)
                if completed > 0 else 0
            )

            progress_bar.set_postfix({
                "loss": f"{loss.item():.3f}",
                "elapsed": f"{elapsed / 60:.1f}m",
                "remaining": f"{remaining / 60:.1f}m"
            })

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model,
                    train_loader,
                    val_loader,
                    device,
                    eval_iter
                )

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                progress_bar.set_postfix({
                    "train_loss": f"{train_loss:.3f}",
                    "val_loss": f"{val_loss:.3f}",
                    "elapsed": f"{elapsed / 60:.1f}m",
                    "remaining": f"{remaining / 60:.1f}m"
                })

            if (
                checkpoint_freq is not None
                and checkpoint_freq > 0
                and global_step > 0
                and global_step % checkpoint_freq == 0
            ):
                torch.save(
                    model.state_dict(),
                    f"{run_name}_model.pth"
                )

                torch.save(
                    {
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch,
                        "global_step": global_step,
                        "tokens_seen": tokens_seen
                    },
                    f"{run_name}_training_checkpoint.pth"
                )

                tqdm.write(
                    f"Checkpoint saved at step {global_step}"
                )

        generate_and_print_sample(
            model,
            tokenizer,
            device,
            start_context
        )

    progress_bar.close()

    print(
        f"Training completed in "
        f"{(time.time() - start_time) / 60:.2f} minutes."
    )

    return train_losses, val_losses, track_tokens_seen

In [19]:
import math

def get_lr(update, warmup_updates, total_updates, peak_lr, min_lr):
    if update < warmup_updates:
        return peak_lr * (update + 1) / warmup_updates
    if update >= total_updates:
        return min_lr
    progress = (update - warmup_updates) / (total_updates - warmup_updates)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))


def train_model_accum(
    model, train_loader, val_loader, optimizer, device,
    accum_steps, total_updates, warmup_updates, peak_lr, min_lr,
    eval_freq, eval_iter, start_context, tokenizer,
    checkpoint_freq, start_update=0, tokens_seen=0,
    run_name="run", eval_train_loader=None
):
    train_losses, val_losses, track_tokens_seen = [], [], []
    update = start_update
    micro = 0
    model.train()
    optimizer.zero_grad()

    def save_checkpoint():
        torch.save(model.state_dict(), f"{run_name}_model.pth")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": 0,
                "global_step": update,
                "tokens_seen": tokens_seen,
                "phase": "accum",
            },
            f"{run_name}_training_checkpoint.pth",
        )
        print(f"Checkpoint saved at update {update}")

    def report():
        train_loss, val_loss = evaluate_model(
            model, eval_train_loader or train_loader, val_loader, device, eval_iter
        )
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        track_tokens_seen.append(tokens_seen)
        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"Update {update:06d} | lr {lr_now:.2e} | "
            f"tokens {tokens_seen / 1e6:.1f}M | "
            f"Train loss {train_loss:.3f}, Validation loss {val_loss:.3f}"
        )

    report()
    done = update >= total_updates
    while not done:
        for input_batch, target_batch in train_loader:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            (loss / accum_steps).backward()
            tokens_seen += input_batch.numel()
            micro += 1
            if micro % accum_steps != 0:
                continue

            lr = get_lr(update, warmup_updates, total_updates, peak_lr, min_lr)
            for group in optimizer.param_groups:
                group["lr"] = lr
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            update += 1

            if update % eval_freq == 0:
                report()
            if update % checkpoint_freq == 0:
                save_checkpoint()
            if update >= total_updates:
                done = True
                break

    save_checkpoint()
    generate_and_print_sample(model, tokenizer, device, start_context)
    return train_losses, val_losses, track_tokens_seen

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
model = Gptmodel(cfg).to(device)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

train_loader, train_eval_loader, val_loader = make_loaders("5M")
x, y = next(iter(train_loader))

model.eval()
with torch.no_grad():
    logits = model(x.to(device))
    loss = nn.functional.cross_entropy(logits.flatten(0, 1), y.to(device).flatten())

print("Logits shape:", tuple(logits.shape))
print("Loss:", round(loss.item(), 3), "| expected near", round(float(np.log(cfg["vocab_size"])), 3))

Parameters: 38,604,800
Logits shape: (4, 1028, 50257)
Loss: 10.989 | expected near 10.825


In [21]:
import json

RUN_NAME = "arch2_chain4_MODEL_2"

for RUN_SIZE in ["5M", "25M", "100M"]:
    torch.manual_seed(SEED)
    model = Gptmodel(cfg).to(device)
    train_loader, train_eval_loader, val_loader = make_loaders(RUN_SIZE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=6e-4, weight_decay=0.1, fused=True)

    total_updates = TOKEN_BUDGET // (MICRO_BS * ACCUM * cfg["context_length"])

    train_losses, val_losses, tokens_seen = train_model_accum(
        model, train_loader, val_loader, optimizer, device,
        accum_steps=ACCUM, total_updates=total_updates,
        warmup_updates=200, peak_lr=6e-4, min_lr=6e-5,
        eval_freq=200, eval_iter=50,
        start_context="Every effort moves you", tokenizer=tokenizer,
        checkpoint_freq=1000, run_name=f"{RUN_NAME}_{RUN_SIZE}",
        eval_train_loader=train_eval_loader,
    )

    model.eval()
    with torch.no_grad():
        final_val = calc_loss_loader(val_loader, model, device)
    print(f"{RUN_SIZE} | val loss {final_val:.3f} | ppl {np.exp(final_val):.1f}")

    with open(f"{RUN_NAME}_{RUN_SIZE}_losses.json", "w") as f:
        json.dump({"train_losses": train_losses, "val_losses": val_losses,
                   "tokens_seen": tokens_seen, "final_val": final_val,
                   "final_ppl": float(np.exp(final_val))}, f)

    del model, optimizer, train_loader, train_eval_loader, val_loader
    torch.cuda.empty_cache()
    

Update 000000 | lr 6.00e-04 | tokens 0.0M | Train loss 10.982, Validation loss 10.981
Update 000200 | lr 6.00e-04 | tokens 3.3M | Train loss 7.212, Validation loss 7.272
Update 000400 | lr 5.98e-04 | tokens 6.6M | Train loss 6.792, Validation loss 6.894
Update 000600 | lr 5.94e-04 | tokens 9.9M | Train loss 6.585, Validation loss 6.725
Update 000800 | lr 5.86e-04 | tokens 13.2M | Train loss 6.426, Validation loss 6.619
Update 001000 | lr 5.76e-04 | tokens 16.4M | Train loss 6.299, Validation loss 6.543
Checkpoint saved at update 1000
Update 001200 | lr 5.62e-04 | tokens 19.7M | Train loss 6.194, Validation loss 6.501
Update 001400 | lr 5.46e-04 | tokens 23.0M | Train loss 6.082, Validation loss 6.447
Update 001600 | lr 5.28e-04 | tokens 26.3M | Train loss 6.003, Validation loss 6.428
Update 001800 | lr 5.07e-04 | tokens 29.6M | Train loss 8.145, Validation loss 8.263
Update 002000 | lr 4.85e-04 | tokens 32.9M | Train loss 8.571, Validation loss 8.605
Checkpoint saved at update 2000
Upd